In [58]:
SMPLX_DIR = 'output/red_test/smplx'


In [59]:
import numpy as np
import torch
import os

In [60]:
def load_word_pose_sequence(smplx_dir, num_frames, start_idx=1):
    """Returns dict of (T, N, 3) axis-angle arrays for one word."""
    orient, pose, lhand, rhand, jaw = [], [], [], [], []
    for f in range(start_idx, start_idx + num_frames):
        npz = dict(np.load(os.path.join(smplx_dir, f'{f:05}_0.npz'), allow_pickle=True))
        orient.append(np.array(npz['global_orient']).reshape(3))
        pose.append(np.array(npz['body_pose']).reshape(21, 3))
        lhand.append(np.array(npz['left_hand_pose']).reshape(15, 3))
        rhand.append(np.array(npz['right_hand_pose']).reshape(15, 3))
        jaw.append(np.array(npz['jaw_pose']).reshape(3))
    return {
        'global_orient': np.stack(orient),
        'body_pose': np.stack(pose),
        'left_hand_pose': np.stack(lhand),
        'right_hand_pose': np.stack(rhand),
        'jaw_pose': np.stack(jaw),
    }

In [61]:
redParams = load_word_pose_sequence(SMPLX_DIR, 132, start_idx=1)

In [97]:
from scipy.spatial.transform import Rotation as R, Slerp
import numpy as np

KEYFRAME_INDICES = [43, 51, 54, 57]   # frame 44,52,55,58 → array idx = frame - start_idx
key_times = np.array([0, 8, 11, 14], dtype=float)

def slerp_array(keyframe_rotvecs, key_times, query_times):
    """keyframe_rotvecs: (K, J, 3) axis-angle -> (T, J, 3)."""
    K, J, _ = keyframe_rotvecs.shape
    out = np.zeros((len(query_times), J, 3))
    for j in range(J):
        rot = R.from_rotvec(keyframe_rotvecs[:, j, :])
        out[:, j, :] = Slerp(key_times, rot)(query_times).as_rotvec()
    return out

kf_global = redParams['global_orient'][KEYFRAME_INDICES].reshape(4, 1, 3)
kf_body   = redParams['body_pose'][KEYFRAME_INDICES]
kf_lhand  = redParams['left_hand_pose'][KEYFRAME_INDICES]
kf_rhand  = redParams['right_hand_pose'][KEYFRAME_INDICES]
kf_jaw    = redParams['jaw_pose'][KEYFRAME_INDICES].reshape(4, 1, 3)                

query_times = np.linspace(0, 3, 15)  # 15 output frames spanning the 4 keyframes

interp_global = slerp_array(kf_global, key_times, query_times)[:, 0, :]
interp_body   = slerp_array(kf_body,   key_times, query_times)
interp_lhand  = slerp_array(kf_lhand,  key_times, query_times)
interp_rhand  = slerp_array(kf_rhand,  key_times, query_times)
interp_jaw    = slerp_array(kf_jaw,    key_times, query_times)[:, 0, :]

In [98]:
SAVE_DIR = 'output/red_test/smplx_interp'

In [ ]:
np.savez(os.path.join(SAVE_DIR, f'{i:05}_0.npz'),
    global_orient=interp_global[i].reshape(1,3),
    body_pose=interp_body[i].reshape(1,21,3),
    left_hand_pose=interp_lhand[i].reshape(1,15,3),
    right_hand_pose=interp_rhand[i].reshape(1,15,3),
    jaw_pose=interp_jaw[i].reshape(1,3),
    betas=betas_fixed, transl=interp_transl[i], expression=expr_fixed)